# Efficient Non-Local Means — exact acceleration demo

The direct implementation explicitly reduces every `K×K` patch for all `S×S` displacements: $O(NS^2K^2)$. The accelerated implementation builds one squared-difference image per displacement and applies a separable running box mean: $O(NS^2)$. Both evaluate the same bias-corrected NLM weights.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
sys.path.insert(0, str(Path('code').resolve()))
from efficient_nlm_core import direct_patch_nlm, fast_exact_nlm, timed

## Load one bundled USC-SIPI image and create a deterministic noisy crop
The 96×96 crop keeps the literal direct reference quick enough for a classroom verification.

In [ ]:
clean_full = np.asarray(Image.open('data/7.1.04.tiff').convert('L'), dtype=np.float64)
clean = clean_full[208:304, 208:304]
sigma_n, h, K, S = 25.0, 15.0, 7, 11
rng = np.random.default_rng(20261029)
noisy = np.clip(clean + rng.normal(0, sigma_n, clean.shape), 0, 255)

## Run both estimators and verify equivalence

In [ ]:
direct, direct_ms = timed(direct_patch_nlm, noisy, sigma_n, h, K, S)
fast, fast_ms = timed(fast_exact_nlm, noisy, sigma_n, h, K, S)
summary = {
    'direct_ms': direct_ms,
    'fast_ms': fast_ms,
    'speedup': direct_ms / fast_ms,
    'max_abs_difference': np.max(np.abs(direct-fast)),
    'PSNR': peak_signal_noise_ratio(clean, fast, data_range=255),
    'SSIM': structural_similarity(clean, fast, data_range=255),
}
summary

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, image, title in zip(axes, [clean, noisy, direct, fast], ['Original', 'Noisy', 'Direct NLM', 'Fast exact NLM']):
    ax.imshow(image, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title); ax.axis('off')
plt.tight_layout()

## Interpretation
A maximum difference near machine precision confirms that the speed-up is computational rather than a change of estimator. The complete native-image and D5/fusion confirmation is saved in `results/` and implemented in `code/best4_final_experiment.py`.